Miramos los valores que hay para cada clase(piedra, papel o tijera).

In [1]:
import sys
sys.path.append("..")

import os

data_path = "../data"  
for carpeta in os.listdir(data_path):
    ruta_carpeta = os.path.join(data_path, carpeta)
    if os.path.isdir(ruta_carpeta):
        n_archivos = len(os.listdir(ruta_carpeta))
        print(f"{carpeta}: {n_archivos} archivos")

paper: 712 archivos
rock: 726 archivos
scissors: 750 archivos


Revisamos el tamaño y el canal que usan estas imagenes.

In [2]:
from PIL import Image

img = Image.open("../data/rock/0bioBZYFCXqJIulm.png")  # Cargamos una imagen
print(img.size, img.mode)

(300, 200) RGB


Redimensionamos y pasamos a tensores.

In [3]:
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

img = Image.open("../data/rock/0bioBZYFCXqJIulm.png").convert("RGB")
tensor_img = transform(img)

print(tensor_img.shape)   # Miramos si ha cambiado las dimensiones
print(tensor_img.min(), tensor_img.max())  # miramos el rango de valores

torch.Size([3, 128, 128])
tensor(0.) tensor(0.9686)


Instanciamos el Dataset.

In [4]:
from src.dataset import RPSDataset

dataset = RPSDataset(root_dir="../data", transform=transform)

print(len(dataset))        # Tamaño del dataset (la suma de todas las imagenes)
img, label = dataset[0]
print(img.shape, label)    # Miramos el formato

2188
torch.Size([3, 128, 128]) 0


In [5]:
from torch.utils.data import random_split

total = len(dataset)
train_size = int(0.8 * total)
test_size = total - train_size

train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

print(len(train_dataset), len(test_dataset))

1750 438


In [6]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [7]:
imagenes, etiquetas = next(iter(train_loader))
print(imagenes.shape)
print(etiquetas.shape)
print(etiquetas)

torch.Size([32, 3, 128, 128])
torch.Size([32])
tensor([1, 2, 0, 0, 2, 0, 1, 2, 2, 2, 2, 0, 0, 1, 0, 0, 2, 2, 2, 1, 1, 0, 1, 2,
        0, 2, 2, 2, 2, 1, 2, 2])


In [8]:
from src.model import SimpleCNN

model = SimpleCNN(num_classes=3)

imagenes, etiquetas = next(iter(train_loader))
salida = model(imagenes)

print(salida.shape)

torch.Size([32, 3])


In [10]:
import torch.nn as nn
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [11]:
optimizer.zero_grad()

salida = model(imagenes)
loss = criterion(salida, etiquetas)

loss.backward()
optimizer.step()

print(loss.item())

1.0935072898864746
